In [0]:
dbutils.fs.ls("/Volumes/adb_caresync/landing/landing_volume/neon_postgres/hospitals/2026-10-17 01:33:38/")

In [0]:
# Create widgets 
dbutils.widgets.text("source_system", "", "Source System")
dbutils.widgets.text("table_name", "", "Table Name")
dbutils.widgets.text("landing_timestamp", "", "Landing Timestamp")
dbutils.widgets.text("source_file_format","","Source File Format")

# Get widget values into variables
source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
if(source_file_format=="csv"):
    landing_df=spark.read.format("csv").option("header","true").option("inferSchema","true").load(f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}")
else:
    landing_df=spark.read.format("parquet").load(f"/Volumes/adb_caresync/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/")

In [0]:
from pyspark.sql import functions as f
updated_df=landing_df.withColumn("landing_timestamp",f.lit(landing_timestamp)).withColumn("insert_timestamp",f.date_format(f.current_timestamp(),"yyyy-MM-dd HH:mm:ss"))

In [0]:
updated_df.write.mode("append").saveAsTable(f"adb_caresync.bronze.{table_name}")